In [1]:
!pip -q install sentence-transformers faiss-cpu transformers sentencepiece


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import faiss
import pickle
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Load the chunks

chunks_path = "./02_rmit_assessment_support_chunks.csv"

chunks_df = pd.read_csv(chunks_path)

print("Chunks loaded:", len(chunks_df))
print("Columns:")
print(chunks_df.columns.tolist())

display(chunks_df.head())

Chunks loaded: 71
Columns:
['chunk_id', 'source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date', 'document_version', 'chunk_number', 'total_chunks', 'chunk_word_count']


,chunk_id,source_id,category,title,section,content,source_url,source_type,collection_date,effective_date,document_version,chunk_number,total_chunks,chunk_word_count
0,EXT-01_0,EXT-01,Extension,Extensions,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-10-05,Not specified,Current webpage,1,1,90
1,EXT-01_1,EXT-01,Extension,Extensions,Assessments eligible for an extension,You can apply for an extension for assessments...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-10-05,Not specified,Current webpage,1,1,50
2,EXT-01_2,EXT-01,Extension,Extensions,How to apply,You must apply at least one working day before...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-10-05,Not specified,Current webpage,1,1,213
3,EXT-01_3,EXT-01,Extension,Extensions,False documents and misleading information,"Creating, submitting or using fraudulent docum...",https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-10-05,Not specified,Current webpage,1,1,90
4,SC-01_4,SC-01,Special Consideration,Special consideration,Special consideration,If unexpected circumstances beyond your contro...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-10-05,Not specified,Current webpage,1,1,46


In [4]:
# Load the FAISS index
index_path = "./02_rmit_assessment_support.faiss"

index = faiss.read_index(index_path)

print("FAISS index loaded successfully.")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", index.d)

FAISS index loaded successfully.
Number of vectors: 71
Vector dimension: 384


In [5]:
#Load the metadata
metadata_path = "./02_rmit_assessment_support_metadata.pkl"

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("Metadata loaded successfully.")
print("Number of metadata records:", len(metadata))

Metadata loaded successfully.
Number of metadata records: 71


In [6]:
#Load the embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2760.68it/s]


Embedding model loaded successfully.


In [7]:
question = "Can I get an extension if I am sick?"

# Convert the question into an embedding
question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

# Normalise it in the same way as our document embeddings
faiss.normalize_L2(question_embedding)

# Retrieve the 5 most relevant chunks
scores, indices = index.search(question_embedding, 5)

print("Question:", question)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunks_df.iloc[idx]

    print("\n------------------------------")
    print("RESULT", rank)
    print("Similarity score:", round(float(score), 4))
    print("Source:", row["source_id"])
    print("Category:", row["category"])
    print("Section:", row["section"])
    print("Chunk ID:", row["chunk_id"])
    print("\nContent:")
    print(row["content"])

Question: Can I get an extension if I am sick?

------------------------------
RESULT 1
Similarity score: 0.4288
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: My School has asked me to fill out a form when requesting an extension. Do I need to do this?
Chunk ID: EAA-02_39

Content:
If you have Equitable Assessment Arrangement for 'Extensions', your Learning Plan is the only evidence you need to provide. You may need to fill out a form if your School requires it, but you do not need to provide a medical certificate or any additional evidence.

------------------------------
RESULT 2
Similarity score: 0.4125
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: How do I ask for an extension?
Chunk ID: EAA-02_34

Content:
If you have the Equitable Assessment Arrangement for ‘Extensions’ within your Equitable Learning Plan, please ask your teacher how you should apply for an extension before the adjustment is due. Different courses use different extensio

In [8]:
test_questions = [
    {
        "question": "Can I get an extension for an assignment?",
        "expected_source": "EXT-01"
    },
    {
        "question": "What should I do if I need more time because of unexpected circumstances?",
        "expected_source": "EXT-01"
    },
    {
        "question": "Am I eligible for special consideration?",
        "expected_source": "SC-01"
    },
    {
        "question": "What evidence do I need for special consideration?",
        "expected_source": "SC-01"
    },
    {
        "question": "How do I apply for an Equitable Assessment Arrangement?",
        "expected_source": "EAA-01"
    },
    {
        "question": "How do I ask for an extension using my Equitable Learning Plan?",
        "expected_source": "EAA-02"
    },
    {
    "question": "Can I get an extension for an exam?",
    "expected_source": "EXT-01"
    },
    {
    "question": "How many days can an assessment extension be?",
    "expected_source": "EXT-01"
    },
    {
    "question": "What should I do if I have technical problems during an online assessment?",
    "expected_source": "SC-01"
    },
    {
    "question": "What documents do I need when registering for disability support?",
    "expected_source": "EAA-02"
    },
    {
    "question": "Can I get an extension for a group assignment?",
    "expected_source": "EAA-02"
    },
    {
    "question": "Can assessment arrangements apply to online tests and practicals?",
    "expected_source": "EAA-02"
    }
]

test_df = pd.DataFrame(test_questions)

display(test_df)

,question,expected_source
0,Can I get an extension for an assignment?,EXT-01
1,What should I do if I need more time because o...,EXT-01
2,Am I eligible for special consideration?,SC-01
3,What evidence do I need for special considerat...,SC-01
4,How do I apply for an Equitable Assessment Arr...,EAA-01
5,How do I ask for an extension using my Equitab...,EAA-02
6,Can I get an extension for an exam?,EXT-01
7,How many days can an assessment extension be?,EXT-01
8,What should I do if I have technical problems ...,SC-01
9,What documents do I need when registering for ...,EAA-02


In [9]:
results = []

for _, test in test_df.iterrows():

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    # Normalise the question embedding
    faiss.normalize_L2(query_embedding)

    # Retrieve the single most relevant chunk
    scores, indices = index.search(query_embedding, 1)

    top_index = indices[0][0]
    top_score = scores[0][0]

    # Find which source the retrieved chunk came from
    retrieved_row = chunks_df.iloc[top_index]
    retrieved_source = retrieved_row["source_id"]

    # Compare retrieved source with expected source
    correct = retrieved_source == test["expected_source"]

    results.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "retrieved_source": retrieved_source,
        "score": round(float(top_score), 4),
        "correct": correct
    })

evaluation_df = pd.DataFrame(results)

display(evaluation_df)

,question,expected_source,retrieved_source,score,correct
0,Can I get an extension for an assignment?,EXT-01,EAA-02,0.5647,False
1,What should I do if I need more time because o...,EXT-01,SC-01,0.3354,False
2,Am I eligible for special consideration?,SC-01,SC-01,0.7528,True
3,What evidence do I need for special considerat...,SC-01,SC-01,0.4730,True
4,How do I apply for an Equitable Assessment Arr...,EAA-01,POL-01,0.7017,False
5,How do I ask for an extension using my Equitab...,EAA-02,EAA-02,0.7889,True
6,Can I get an extension for an exam?,EXT-01,EAA-02,0.6186,False
7,How many days can an assessment extension be?,EXT-01,EXT-01,0.8408,True
8,What should I do if I have technical problems ...,SC-01,SC-01,0.7390,True
9,What documents do I need when registering for ...,EAA-02,EAA-02,0.7236,True


In [10]:
results_top3 = []

for _, test in test_df.iterrows():

    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # Retrieve top 3 chunks
    scores, indices = index.search(query_embedding, 3)

    retrieved_sources = [
        chunks_df.iloc[idx]["source_id"]
        for idx in indices[0]
    ]

    # Is the expected source anywhere in the top 3?
    correct_top3 = test["expected_source"] in retrieved_sources

    results_top3.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "top_3_sources": retrieved_sources,
        "correct_top3": correct_top3
    })

evaluation_top3_df = pd.DataFrame(results_top3)

display(evaluation_top3_df)

,question,expected_source,top_3_sources,correct_top3
0,Can I get an extension for an assignment?,EXT-01,"[EAA-02, EAA-02, EAA-02]",False
1,What should I do if I need more time because o...,EXT-01,"[SC-01, EXT-01, EAA-02]",True
2,Am I eligible for special consideration?,SC-01,"[SC-01, SC-01, POL-01]",True
3,What evidence do I need for special considerat...,SC-01,"[SC-01, SC-01, EAA-02]",True
4,How do I apply for an Equitable Assessment Arr...,EAA-01,"[POL-01, EAA-01, EAA-02]",True
5,How do I ask for an extension using my Equitab...,EAA-02,"[EAA-02, EAA-02, EAA-02]",True
6,Can I get an extension for an exam?,EXT-01,"[EAA-02, EAA-02, EAA-02]",False
7,How many days can an assessment extension be?,EXT-01,"[EXT-01, POL-01, EAA-02]",True
8,What should I do if I have technical problems ...,SC-01,"[SC-01, SC-01, SC-01]",True
9,What documents do I need when registering for ...,EAA-02,"[EAA-02, EAA-02, SC-01]",True


In [11]:
question = "Can I get an extension for an assignment?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

# Retrieve top 5
scores, indices = index.search(query_embedding, 5)

print("Question:", question)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunks_df.iloc[idx]

    print("\n------------------------------")
    print("Rank:", rank)
    print("Source:", row["source_id"])
    print("Section:", row["section"])
    print("Score:", round(float(score), 4))
    print("Content:", row["content"][:400])

Question: Can I get an extension for an assignment?

------------------------------
Rank: 1
Source: EAA-02
Section: My teacher has directed me to apply for special consideration even though I have the Equitable Assessment Arrangement for extensions. What do I do?
Score: 0.5647
Content: If it is your first extension, you do not need to apply for special consideration. Direct your teacher to your Equitable Learning Plan where this is clearly stated. If you aren’t able to implement your extension request, contact ELS for assistance. If you require a second extension, you will need to apply for special consideration .

------------------------------
Rank: 2
Source: EAA-02
Section: How many days can I request for an extension (7 days, 14 days, etc)?
Score: 0.5171
Content: If you have the Equitable Assessment Arrangement for ‘Extensions’ within your Equitable Learning Plan, you can request an extension for individual assessments (eg. essays, assignments, portfolios and projects). The length 

In [13]:
results_top5 = []

for _, test in test_df.iterrows():

    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # Retrieve top 5 chunks
    scores, indices = index.search(query_embedding, 5)

    retrieved_sources = [
        chunks_df.iloc[idx]["source_id"]
        for idx in indices[0]
    ]

    correct_top5 = test["expected_source"] in retrieved_sources

    results_top5.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "top_5_sources": retrieved_sources,
        "correct_top5": correct_top5
    })

evaluation_top5_df = pd.DataFrame(results_top5)

display(evaluation_top5_df)

,question,expected_source,top_5_sources,correct_top5
0,Can I get an extension for an assignment?,EXT-01,"[EAA-02, EAA-02, EAA-02, EXT-01, EAA-02]",True
1,What should I do if I need more time because o...,EXT-01,"[SC-01, EXT-01, EAA-02, SC-01, POL-01]",True
2,Am I eligible for special consideration?,SC-01,"[SC-01, SC-01, POL-01, POL-01, EAA-02]",True
3,What evidence do I need for special considerat...,SC-01,"[SC-01, SC-01, EAA-02, SC-01, SC-01]",True
4,How do I apply for an Equitable Assessment Arr...,EAA-01,"[POL-01, EAA-01, EAA-02, EAA-02, POL-01]",True
5,How do I ask for an extension using my Equitab...,EAA-02,"[EAA-02, EAA-02, EAA-02, EAA-02, EAA-02]",True
6,Can I get an extension for an exam?,EXT-01,"[EAA-02, EAA-02, EAA-02, EXT-01, EAA-02]",True
7,How many days can an assessment extension be?,EXT-01,"[EXT-01, POL-01, EAA-02, EAA-02, EAA-02]",True
8,What should I do if I have technical problems ...,SC-01,"[SC-01, SC-01, SC-01, SC-01, POL-01]",True
9,What documents do I need when registering for ...,EAA-02,"[EAA-02, EAA-02, SC-01, EAA-02, EAA-02]",True


In [14]:
top1_accuracy = evaluation_df["correct"].mean()
top3_accuracy = evaluation_top3_df["correct_top3"].mean()
top5_accuracy = evaluation_top5_df["correct_top5"].mean()

print("Retrieval Evaluation")
print("--------------------")
print(f"Top-1 Accuracy: {top1_accuracy:.1%}")
print(f"Top-3 Accuracy: {top3_accuracy:.1%}")
print(f"Top-5 Accuracy: {top5_accuracy:.1%}")

Retrieval Evaluation
--------------------
Top-1 Accuracy: 66.7%
Top-3 Accuracy: 83.3%
Top-5 Accuracy: 100.0%


In [15]:
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

print("Original content:")
print(chunks_df.iloc[0]["content"][:500])

print("\n--------------------")
print("New embedding text:")
print(chunks_df.iloc[0]["embedding_text"][:700])

Original content:
If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seven days , you must apply for special consideration . However, if you have an equitable assessment arrangement which allows for the negotiation of submission dates with academic/teaching staff, extensions o

--------------------
New embedding text:
Extension. If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.. If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extensio

In [17]:
improved_embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

print("Number of chunks:", len(chunks_df))
print("Improved embedding shape:", improved_embeddings.shape)

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.31it/s]

Number of chunks: 71
Improved embedding shape: (71, 384)


In [18]:
# Make a copy before normalising
improved_embeddings_normalized = improved_embeddings.copy()

# Normalise for cosine similarity
faiss.normalize_L2(improved_embeddings_normalized)

# Create a new FAISS index
dimension = improved_embeddings_normalized.shape[1]
improved_index = faiss.IndexFlatIP(dimension)

# Add improved embeddings
improved_index.add(improved_embeddings_normalized)

print("Improved FAISS index created successfully")
print("Embedding dimensions:", dimension)
print("Vectors stored:", improved_index.ntotal)

Improved FAISS index created successfully
Embedding dimensions: 384
Vectors stored: 71


In [19]:
def evaluate_index(search_index, test_df):
    results = []

    for _, test in test_df.iterrows():

        query_embedding = embedding_model.encode(
            [test["question"]],
            convert_to_numpy=True
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        # Retrieve top 5
        scores, indices = search_index.search(query_embedding, 5)

        retrieved_sources = [
            chunks_df.iloc[idx]["source_id"]
            for idx in indices[0]
        ]

        results.append({
            "question": test["question"],
            "expected_source": test["expected_source"],
            "top_1_source": retrieved_sources[0],
            "correct_top1": test["expected_source"] == retrieved_sources[0],
            "correct_top3": test["expected_source"] in retrieved_sources[:3],
            "correct_top5": test["expected_source"] in retrieved_sources
        })

    return pd.DataFrame(results)


# Evaluate both versions
baseline_12 = evaluate_index(index, test_df)
improved_12 = evaluate_index(improved_index, test_df)

print("BASELINE - Content Only")
print("-----------------------")
print(f"Top-1: {baseline_12['correct_top1'].mean():.1%}")
print(f"Top-3: {baseline_12['correct_top3'].mean():.1%}")
print(f"Top-5: {baseline_12['correct_top5'].mean():.1%}")

print("\nIMPROVED - Category + Section + Content")
print("---------------------------------------")
print(f"Top-1: {improved_12['correct_top1'].mean():.1%}")
print(f"Top-3: {improved_12['correct_top3'].mean():.1%}")
print(f"Top-5: {improved_12['correct_top5'].mean():.1%}")

BASELINE - Content Only
-----------------------
Top-1: 66.7%
Top-3: 83.3%
Top-5: 100.0%

IMPROVED - Category + Section + Content
---------------------------------------
Top-1: 83.3%
Top-3: 91.7%
Top-5: 100.0%


In [20]:
failed_improved = improved_12[
    improved_12["correct_top1"] == False
]

display(failed_improved[
    ["question", "expected_source", "top_1_source"]
])

,question,expected_source,top_1_source
1,What should I do if I need more time because o...,EXT-01,SC-01
4,How do I apply for an Equitable Assessment Arr...,EAA-01,POL-01


In [21]:
improved_results = []

for _, test in test_df.iterrows():

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # Retrieve top 5 from improved index
    scores, indices = improved_index.search(query_embedding, 5)

    retrieved_sources = [
        chunks_df.iloc[idx]["source_id"]
        for idx in indices[0]
    ]

    improved_results.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "top_1_source": retrieved_sources[0],
        "top_3_sources": retrieved_sources[:3],
        "top_5_sources": retrieved_sources,
        "correct_top1": test["expected_source"] == retrieved_sources[0],
        "correct_top3": test["expected_source"] in retrieved_sources[:3],
        "correct_top5": test["expected_source"] in retrieved_sources
    })

improved_evaluation_df = pd.DataFrame(improved_results)

display(improved_evaluation_df)

print("\nImproved Retrieval Evaluation")
print("-----------------------------")
print(f"Top-1 Accuracy: {improved_evaluation_df['correct_top1'].mean():.1%}")
print(f"Top-3 Accuracy: {improved_evaluation_df['correct_top3'].mean():.1%}")
print(f"Top-5 Accuracy: {improved_evaluation_df['correct_top5'].mean():.1%}")

,question,expected_source,top_1_source,top_3_sources,top_5_sources,correct_top1,correct_top3,correct_top5
0,Can I get an extension for an assignment?,EXT-01,EXT-01,"[EXT-01, EAA-02, EAA-02]","[EXT-01, EAA-02, EAA-02, EAA-02, EAA-02]",True,True,True
1,What should I do if I need more time because o...,EXT-01,SC-01,"[SC-01, EXT-01, EAA-02]","[SC-01, EXT-01, EAA-02, SC-01, SC-01]",False,True,True
2,Am I eligible for special consideration?,SC-01,SC-01,"[SC-01, SC-01, SC-01]","[SC-01, SC-01, SC-01, POL-01, SC-01]",True,True,True
3,What evidence do I need for special considerat...,SC-01,SC-01,"[SC-01, SC-01, SC-01]","[SC-01, SC-01, SC-01, SC-01, SC-01]",True,True,True
4,How do I apply for an Equitable Assessment Arr...,EAA-01,POL-01,"[POL-01, EAA-02, EAA-02]","[POL-01, EAA-02, EAA-02, EAA-01, EAA-02]",False,False,True
5,How do I ask for an extension using my Equitab...,EAA-02,EAA-02,"[EAA-02, EAA-02, EAA-02]","[EAA-02, EAA-02, EAA-02, EAA-02, EAA-02]",True,True,True
6,Can I get an extension for an exam?,EXT-01,EXT-01,"[EXT-01, EAA-02, EXT-01]","[EXT-01, EAA-02, EXT-01, EAA-02, EAA-02]",True,True,True
7,How many days can an assessment extension be?,EXT-01,EXT-01,"[EXT-01, EAA-02, POL-01]","[EXT-01, EAA-02, POL-01, EAA-02, EXT-01]",True,True,True
8,What should I do if I have technical problems ...,SC-01,SC-01,"[SC-01, SC-01, SC-01]","[SC-01, SC-01, SC-01, EAA-02, EAA-02]",True,True,True
9,What documents do I need when registering for ...,EAA-02,EAA-02,"[EAA-02, SC-01, EAA-02]","[EAA-02, SC-01, EAA-02, EAA-02, EAA-02]",True,True,True



Improved Retrieval Evaluation
-----------------------------
Top-1 Accuracy: 83.3%
Top-3 Accuracy: 91.7%
Top-5 Accuracy: 100.0%


In [22]:
question = "How do I apply for an Equitable Assessment Arrangement?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

scores, indices = improved_index.search(query_embedding, 10)

print("Question:", question)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunks_df.iloc[idx]

    print("\n------------------------------")
    print("Rank:", rank)
    print("Source:", row["source_id"])
    print("Category:", row["category"])
    print("Section:", row["section"])
    print("Score:", round(float(score), 4))

Question: How do I apply for an Equitable Assessment Arrangement?

------------------------------
Rank: 1
Source: POL-01
Category: Policy
Section: Equitable Assessment Arrangements
Score: 0.7214

------------------------------
Rank: 2
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: My teacher has directed me to apply for special consideration even though I have the Equitable Assessment Arrangement for extensions. What do I do?
Score: 0.7083

------------------------------
Rank: 3
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: Do my Equitable Assessment Arrangements apply to in-class and online tests, and practicals?
Score: 0.7026

------------------------------
Rank: 4
Source: EAA-01
Category: Equitable Assessment Arrangements
Section: What is an EAA?
Score: 0.6644

------------------------------
Rank: 5
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: What information do I need to provide?
Score: 0.661

----------------------

In [24]:
question = "How do I apply for an Equitable Assessment Arrangement?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

# Search all 127 chunks
scores, indices = improved_index.search(query_embedding, len(chunks_df))

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunks_df.iloc[idx]

    if row["source_id"] == "EAA-01":
        print("First EAA-01 result")
        print("Rank:", rank)
        print("Section:", row["section"])
        print("Score:", round(float(score), 4))
        print("Content:", row["content"][:500])
        break

First EAA-01 result
Rank: 4
Section: What is an EAA?
Score: 0.6644
Content: An Equitable Assessment Arrangement (EAA) is an adjustment to the standard conditions or format of an assessment for students with a disability, long-term illness and/or mental health condition. An EAA may also be available to primary carers of a close family member with a disability, long-term illness and/or mental health condition, where this means that an adjustment is needed to participate in an assessment activity. EAAs can apply to any formal assessment task described in the course guide. 


In [25]:
question = "How do I apply for an Equitable Assessment Arrangement?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

scores, indices = improved_index.search(
    query_embedding,
    len(chunks_df)
)

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]),
    start=1
):
    row = chunks_df.iloc[idx]

    if (
        row["source_id"] == "EAA-01"
        and "How to apply" in str(row["section"])
    ):
        print("EAA-01 How to apply")
        print("Rank:", rank)
        print("Section:", row["section"])
        print("Score:", round(float(score), 4))
        print("Content:", row["content"][:700])
        break

EAA-01 How to apply
Rank: 6
Section: How to apply
Score: 0.654
Content: To access an EAA, you must register with Equitable Learning Services (ELS) . It is recommended that you register at least six weeks before a major timed assessment or four weeks before your first assessment task. If you register after this time, we may not be able to offer a full range of assistance. ELS will require supporting documentation from a health practitioner (e.g. a doctor, psychologist or social worker) who is treating your condition and is registered with a recognised professional body.


In [26]:
def retrieve_context(question, search_index, top_k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = search_index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = []

    for idx, score in zip(indices[0], scores[0]):

        row = chunks_df.iloc[idx]

        retrieved_chunks.append({
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "source_url": row["source_url"],
            "score": round(float(score), 4)
        })

    return retrieved_chunks

In [27]:
question = "Can I get an extension for an assignment?"

retrieved_context = retrieve_context(
    question,
    improved_index,
    top_k=1
)

In [28]:
for rank, chunk in enumerate(retrieved_context, start=1):
    print(f"\n--- Rank {rank} ---")
    print("Source:", chunk["source_id"])
    print("Section:", chunk["section"])
    print("Score:", chunk["score"])
    print("Content:", chunk["content"][:500])


--- Rank 1 ---
Source: EXT-01
Section: Assessments eligible for an extension
Score: 0.5881
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for special consideration .


In [29]:
def build_context(retrieved_chunks):

    context_parts = []

    for rank, chunk in enumerate(retrieved_chunks, start=1):

        context_part = f"""
Source {rank}
Source ID: {chunk['source_id']}
Category: {chunk['category']}
Section: {chunk['section']}
Content: {chunk['content']}
URL: {chunk['source_url']}
"""

        context_parts.append(context_part)

    return "\n".join(context_parts)


context_text = build_context(retrieved_context)

print(context_text)


Source 1
Source ID: EXT-01
Category: Extension
Section: Assessments eligible for an extension
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for special consideration .
URL: https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions



In [30]:
def build_prompt(question, context):

    prompt = f"""
You are an assistant helping RMIT students understand assessment support.

Answer the student's question using only the information provided in the context below.

Rules:
- Do not invent information.
- If the context does not contain enough information, say that you do not have enough information.
- Answer in 2 to 3 complete sentences.
- Explain the answer using the information in the context.
- Distinguish general assessment extension information from Equitable Learning Plan (ELP) arrangements where relevant.
- Base the answer only on the retrieved RMIT information.

Student question:
{question}

Context:
{context}

Answer:
"""

    return prompt


rag_prompt = build_prompt(
    question,
    context_text
)

print(rag_prompt)


You are an assistant helping RMIT students understand assessment support.

Answer the student's question using only the information provided in the context below.

Rules:
- Do not invent information.
- If the context does not contain enough information, say that you do not have enough information.
- Answer in 2 to 3 complete sentences.
- Explain the answer using the information in the context.
- Distinguish general assessment extension information from Equitable Learning Plan (ELP) arrangements where relevant.
- Base the answer only on the retrieved RMIT information.

Student question:
Can I get an extension for an assignment?

Context:

Source 1
Source ID: EXT-01
Category: Extension
Section: Assessments eligible for an extension
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility 

In [31]:
from transformers import AutoTokenizer

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("FLAN-T5 tokenizer loaded successfully.")

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KRUSHIL\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hu

FLAN-T5 tokenizer loaded successfully.


In [32]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

print("FLAN-T5 model loaded successfully.")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 811.15it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 model loaded successfully.


In [33]:
inputs = tokenizer(
    rag_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

rag_answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("RAG answer:")
print(rag_answer)

RAG answer:
Can I get an extension for an assignment?


In [34]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.18.0


In [35]:
print("Chunks:", len(chunks_df))
print("Improved index vectors:", improved_index.ntotal)
print("Question:", question)
print("Prompt ready:", len(rag_prompt) > 0)

Chunks: 71
Improved index vectors: 71
Question: Can I get an extension for an assignment?
Prompt ready: True


In [36]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.14.1+cpu
Transformers: 5.18.0
CUDA available: False


In [37]:
from transformers import AutoTokenizer

print("AutoTokenizer imported successfully")

AutoTokenizer imported successfully


In [38]:
print("Kernel is working")

Kernel is working


In [39]:
import huggingface_hub

print("Hugging Face Hub imported successfully")
print("Version:", huggingface_hub.__version__)

Hugging Face Hub imported successfully
Version: 1.33.0


In [40]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="google/flan-t5-small",
    filename="config.json"
)

print("Download successful")
print(file_path)

Download successful
C:\Users\KRUSHIL\.cache\huggingface\hub\models--google--flan-t5-small\snapshots\0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab\config.json


In [41]:
import sentencepiece as spm

print("SentencePiece imported successfully")
print("Version:", spm.__version__)

SentencePiece imported successfully
Version: 0.2.2


In [42]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

base_model_name = "google/flan-t5-base"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

print("FLAN-T5-base loaded successfully.")

Loading weights: 100%|██████████| 282/282 [00:01<00:00, 281.89it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5-base loaded successfully.


In [43]:
print(type(tokenizer))
print(type(model))

<class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>
<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [44]:
test_prompt = "What is an assessment extension?"

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated answer:")
print(answer)

Generated answer:
a re-examine


In [45]:
encoded = tokenizer(
    rag_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

print("Tokens sent to model:", encoded["input_ids"].shape[1])

decoded_input = tokenizer.decode(
    encoded["input_ids"][0],
    skip_special_tokens=True
)

print("\nActual prompt received by model:\n")
print(decoded_input)

Tokens sent to model: 269

Actual prompt received by model:

You are an assistant helping RMIT students understand assessment support. Answer the student's question using only the information provided in the context below. Rules: - Do not invent information. - If the context does not contain enough information, say that you do not have enough information. - Answer in 2 to 3 complete sentences. - Explain the answer using the information in the context. - Distinguish general assessment extension information from Equitable Learning Plan (ELP) arrangements where relevant. - Base the answer only on the retrieved RMIT information. Student question: Can I get an extension for an assignment? Context: Source 1 Source ID: EXT-01 Category: Extension Section: Assessments eligible for an extension Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assess